# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Citation: {metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by @id
record_sets = []
if hasattr(metadata, 'record_sets'):
    print("Available record sets:")
    for rs in metadata.record_sets:
        record_sets.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs)
        name = rs.get('name', '') if isinstance(rs, dict) else ''
        print(f"  - @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs}" + (f", name: {name}" if name else ''))
else:
    print("No record sets found in metadata.")
# For the FAIR^2 dataset, we fall back to dataset.record_sets() if record_sets field is empty
if not record_sets:
    try:
        for rs in dataset.record_sets():
            print(f"Found record set: @id = {rs['@id']}, name = {rs['name']}")
            record_sets.append(rs['@id'])
    except Exception as e:
        print("Could not enumerate record sets (may require specific Croissant field structure). Error:", e)
if not record_sets:
    # If no record sets found, print a message.
    print("No record sets enumerated. Please check the Croissant schema or mlcroissant version.")

### View Example Records in Each Record Set
Print the first record from each available record set using its `@id`.

In [ ]:
for rs_id in record_sets:
    print(f"---\nSample record from record set {rs_id}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if i == 0:
                break
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** For this dataset, record set `@id`s may be auto-discovered as shown above. Below, we demonstrate extracting all record sets.

In [ ]:
dataframes = {}
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"DataFrame loaded for record set @id: {rs_id}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for record set {rs_id}")
    except Exception as e:
        print(f"Could not extract DataFrame for record set {rs_id}: {e}")

If you know a specific record set you're interested in, use its `@id` and field `@id` values below.

In [ ]:
# Example: Choose a record set by its @id for further EDA.
if record_sets:
    selected_record_set = record_sets[0]
    df = dataframes[selected_record_set]
    print(f"Analyzing record set: {selected_record_set}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA on protein abundance data
# You may adjust <numeric_field_id> and <group_field_id> as per the displayed columns above.
import numpy as np
import warnings

if record_sets:
    # Replace these with real column names (IDs) if necessary
    df = dataframes[selected_record_set]
    print(f"Available columns for EDA: {df.columns.tolist()}")
    # Try to guess a numeric field (e.g. 'abundance', 'molecular_weight', etc.)
    # We'll pick first numeric column found.
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print('No numeric field detected in this record set.')
    else:
        print(f'Using numeric field for EDA: {numeric_field}')
        threshold = np.nanmean(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > mean ({threshold:.3f}):")
        display(filtered_df.head())
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # ignore SettingWithCopyWarning
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records (z-score normalization):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to pick a group field (e.g. 'description', 'protein_class', etc.)
        group_field = None
        possible_group_fields = ['description', 'group', 'type', 'class', 'sample', 'accession']
        for col in df.columns:
            for hint in possible_group_fields:
                if hint in col:
                    group_field = col
                    break
            if group_field:
                break
        # Otherwise default to first non-numeric column
        if not group_field:
            for col in df.columns:
                if not pd.api.types.is_numeric_dtype(df[col]):
                    group_field = col
                    break
        if group_field and group_field in df.columns:
            print(f"Grouping filtered data by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            display(grouped_df.head())
        else:
            print("No appropriate group field found for grouping.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field is not None:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {selected_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped, show top-10 means by group
    if group_field and group_field in df.columns:
        means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10,4))
        sns.barplot(y=means.index, x=means.values, orient='h')
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key findings:**
- Loaded the Croissant metadata for the FAIR^2 dataset.
- Discovered record set(s) via their `@id` and explored their data structure.
- Filtered and normalized numeric fields for exploratory analysis.
- Visualized numeric field distributions and group-level means where available.

Further analysis may involve advanced filtering, feature engineering, or building predictive models using the loaded data. Refer to the Croissant schema and mlcroissant documentation for dataset-specific field IDs, semantics, and recommended usages.